# 04 - Pairwise judging with Kaggle models  (Task 7) + report tables (Task 8)

**How to create this notebook (important):** the free judge-model proxy only exists in notebooks created from
**https://www.kaggle.com/benchmarks/tasks/new**. That page opens a new notebook with `kaggle-benchmarks`
pre-installed and the proxy credentials set. In that notebook: File -> Import Notebook -> upload this `.ipynb`
(or paste the cells). Then **attach the output of notebook 03** as input. No GPU is needed. Internet = On.

The judge reads only the saved pair files, sends 5 pairs per request, sleeps >= 5 s between attempts, retries each
batch up to 3 times, writes the judged file after every batch, logs failures separately, and resumes from existing
judgments. To resume after a disconnect, also attach this notebook's previous version output. The cell
"resume" copies earlier judgments in.

In [ ]:
# ---- EDIT THIS: your GitHub fork of the starter repo with this code pushed to it ----
REPO_URL = "https://github.com/Ibrahim2424777/CS5326-PA2.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/cs5326-pa2"

JUDGE_MODEL = "google/gemini-3.6-flash"   # if missing, the script falls back to another Gemini flash model and says so
JUDGE_OPTIONAL_SFT_VS_BASE = False        # extra 300 pairs, only for your own analysis

In [ ]:
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
import runpy
from smalign.notebook import sh, start_bg, wait_bg, find_input, copy_run_metadata, zip_dir, disk_report

In [ ]:
# Which judge models does the proxy offer in this notebook?
import kaggle_benchmarks as kbench
print("available:", sorted(getattr(kbench, "llms", {}) or {}))
print("default llm:", getattr(kbench.llm, "name", None), "| default judge:", getattr(getattr(kbench, "judge_llm", None), "name", None))
print("LLMS_AVAILABLE env:", os.environ.get("LLMS_AVAILABLE"))

In [ ]:
# Bring in notebook 03's outputs (pairs, generations, metrics, plots); never regenerate model outputs here.
import shutil, glob
src = os.path.dirname(os.path.dirname(find_input("outputs/eval/dpo_uf_vs_sft_pairs.jsonl")))
shutil.copytree(src, "outputs", dirs_exist_ok=True)
# resume: copy judgments from a previous version of THIS notebook if attached
for path in glob.glob("/kaggle/input/**/outputs/eval/**/*_judged*.json*", recursive=True):
    rel = path.split("/outputs/eval/", 1)[1]
    os.makedirs(os.path.dirname(f"outputs/eval/{rel}") or ".", exist_ok=True)
    shutil.copy2(path, f"outputs/eval/{rel}")
    print("resumed from", path)
sh("ls -la outputs/eval")

## Run the judge (in-process, so it uses this kernel's model-proxy credentials)

In [ ]:
def judge(name, pairs_rel):
    args = ["--pairs_path", f"outputs/eval/{pairs_rel}_pairs.jsonl", "--output_path", f"outputs/eval/{pairs_rel}_judged.jsonl",
            "--judge_model", JUDGE_MODEL, "--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3", "--reuse_judgments"]
    print(f"===== judging {name}")
    old = sys.argv
    sys.argv = ["scripts/judge_pairwise_kaggle.py", *args]
    try:
        runpy.run_path("scripts/judge_pairwise_kaggle.py", run_name="__main__")
    finally:
        sys.argv = old

judge("UltraFeedback-DPO vs SFT on AlpacaEval-300", "dpo_uf_vs_sft")
judge("HH-RLHF-DPO vs SFT on the HH eval set", "dpo_hh_vs_sft")
judge("LoRA-DPO vs SmolLM2-Instruct on persona prompts", "lora_dpo_vs_base")
if JUDGE_OPTIONAL_SFT_VS_BASE and os.path.exists("outputs/eval/optional/sft_vs_base_pairs.jsonl"):
    judge("SFT vs base (optional)", "optional/sft_vs_base")

Re-run the cell above if some pairs ended as `failed`/`unknown`: `--reuse_judgments` only re-sends those.

In [ ]:
sh("cat outputs/eval/*_summary.json | head -c 4000")
sh("python scripts/summarize_results.py")
from IPython.display import Markdown, display
display(Markdown(open("outputs/report_assets/results.md", encoding="utf-8").read()[:20000]))
zip_dir("outputs", "/kaggle/working/final_artifacts.zip")
print("DONE. Download final_artifacts.zip, unzip it into the repo root (it contains outputs/...), commit it, and write the report.")